In [ ]:
library(Seurat)
library(reticulate)
library(anndata)
options(future.globals.maxSize = 1 * 1024^3)

xenium_name <- "Region1"
TABLE       <- "REGION1_TABLES"
xenium_dir <- "/Volumes/ProstateCancerEvoMain/raw/xenium/output-XETG00283__0037793__Region_1__20241204__160313"

cell_guide <- read.csv(file.path(
  "/Volumes/ProstateCancerEvoMain/dbs/Completed/AllRegions/CellTypes",
  paste0(xenium_name, ".raw.annotated.V2.guide.csv")
), stringsAsFactors=FALSE)

cell_guide

base_dir   <- file.path("/Volumes/ProstateCancerEvoMain/dbs/Ongoing", xenium_name, TABLE)
h5ad_file  <- file.path(
  base_dir,
  paste0(xenium_name, "_Xenium_Phen_HE_Integrated.Protein_PhenCycTable.V1.h5ad")
)
rds_file   <- file.path(
  base_dir,
  paste0(xenium_name, "_Xenium_Phen_HE_Integrated.Protein_PhenCycTable.V1.rds")
)

data <- read_h5ad(h5ad_file)

seu_phen <- CreateSeuratObject(
  counts    = t(as.matrix(data$X)),
  meta.data = data$obs
)

saveRDS(seu_phen, file = rds_file)

In [ ]:
"_____________________________________________________ Add Xenium from raw output folder _____________________________________________________________________"


adata_xenium <- LoadXenium(
  data.dir            = xenium_dir,    # path to folder with Xenium CSVs/parquets
  fov                 = "fov",         # name to assign to this field of view
  assay               = "Xenium",      # assay slot name
  #mols.qv.threshold   = 20,            # quality‐value cutoff for transcripts
  cell.centroids      = TRUE,          # load cell centroid coords
  molecule.coordinates= FALSE          # skip loading raw molecule pixels
)





In [ ]:

"________________________________________________________ Change the names of PhenCyc with Xenium __________________________________________________________________"
adata_xenium
adata_phen <- seu_phen

new_names <- setNames(colnames(adata_xenium), colnames(adata_phen))
adata_phen <- RenameCells(adata_phen, new.names = new_names)


seu <- adata_xenium
DefaultAssay(seu) <- "Xenium"

seu[["PhenCyc"]] <- adata_phen[["RNA"]]


In [ ]:
" _____________________________________________________ Define Cell Guides Previous Annotations ____________________________________________________________"

rownames(cell_guide) <- cell_guide$cell_id
rownames(cell_guide)

cell_guide$cell_id <- NULL

all(rownames(seu@meta.data) %in% rownames(cell_guide))

seu <- AddMetaData(
  object   = seu,
  metadata = cell_guide
)

In [ ]:

"________________________________________________________ Concat PhenCyc + Xenium in one Obj and Preprocess [PCA + Variable Genes + Scaling] __________________________________________________________________"

for (assay in c("Xenium","PhenCyc")) {
  DefaultAssay(seu) <- assay
  seu <- NormalizeData(seu, verbose = FALSE)
  
  seu <- FindVariableFeatures(seu, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
  
  seu <- ScaleData(seu, features = VariableFeatures(seu), verbose = FALSE)
  
  
  seu <- RunPCA(seu,
                features = VariableFeatures(seu),
                reduction.name = paste0("pca_", assay),
                verbose = FALSE)
}



In [ ]:

"________________________________________________________ Holy - WNN Part  __________________________________________________________________"

seu <- FindMultiModalNeighbors(
  object = seu,
  reduction.list    = list("pca_Xenium", "pca_PhenCyc"),
  dims.list         = list(1:20,       1:20),
  modality.weight.name = "PhenCyc.weight"
)



In [ ]:
"________________________________________________________ Holy - WNN Part  __________________________________________________________________"

seu <- RunUMAP(seu, nn.name = "weighted.nn", reduction.name = "wnn.umap",
               reduction.key = "wnnUMAP_")

seu <- FindClusters(seu, graph.name = "wsnn", algorithm = 3, resolution = 0.5)



In [ ]:
"________________________________________________________ Annotating Clusters  __________________________________________________________________"


# Assuming your clusters are stored in seu$seurat_clusters
Idents(seu) <- "seurat_clusters"

# 3. Find markers using only the Xenium assay
markers_xenium <- FindAllMarkers(
  object          = seu,
  assay           = "Xenium",        # explicitly use Xenium assay
  slot            = "data",          # use normalized data slot
  only.pos        = TRUE,
  min.pct         = 0.25,
  logfc.threshold = 0.25
)

library(tidyverse)   # loads dplyr + ggplot2 + etc.

# Inspect the top 5 markers per cluster
top5 <- markers_xenium %>% 
  group_by(cluster) %>% 
  slice_max(order_by = avg_log2FC, n = 5)
print(top5)



In [ ]:
"________________________________________________________ Plot Clusters  __________________________________________________________________"

p_clusters <- DimPlot(
  object    = seu,
  reduction = "wnn.umap",
  group.by  = "seurat_clusters",  # or replace with "wsnn" if that's your column
  label     = TRUE
) + ggtitle("WNN‐UMAP Clusters (resolution = 0.5)")

p_clusters



In [ ]:

"________________________________________________________ Annotating Clusters by Guide  __________________________________________________________________"

# 1. Coerce to character so we can fill in NAs
seu$assigned_celltype_L0 <- as.character(seu$assigned_celltype_L0)
# 2. Fill missing/empty
seu$assigned_celltype_L0[is.na(seu$assigned_celltype_L0) | seu$assigned_celltype_L0 == ""] <- "Unknown"
# 3. Turn back into a factor (optional: specify levels if you want a custom order)
seu$assigned_celltype_L0 <- factor(seu$assigned_celltype_L0)

# 4. Now DimPlot will work
p_cell_type_all <- DimPlot(
  object    = seu,
  reduction = "wnn.umap",
  group.by  = "assigned_celltype_L0",
  label     = TRUE,
  label.size= 4,
  raster    = FALSE
)
print(p_cell_type_all)

#seu$NF_H_vs_Others <- ifelse(
#  seu$cell_type == "NF-H",   # replace with your exact CD8 label
#  "NF-H",
#  "Other"
#)

#p_phen_assigned <- DimPlot(
#  seu,
#  reduction = "wnn.umap", 
#  group.by  = "NF_H_vs_Others",
#  cols      = c("NF-H" = "red", "Other" = "blue"),
#  pt.size   = 0.5
#) 
#p_phen_assigned



In [ ]:
"________________________________________________________ CrossTab Set   __________________________________________________________________"

library(pheatmap)

meta <- seu@meta.data

ct_tab <- table(
  CellType          = meta$cell_type,
  AssignedCellType  = meta$assigned_celltype
)

# 4. (Optional) Normalize by row to show proportions
 ct_tab <- prop.table(ct_tab, margin = 2)

# 5. Plot heatmap
pheatmap(
  mat            = ct_tab,
  cluster_rows   = FALSE,
  cluster_cols   = FALSE,
  display_numbers= TRUE,
  fontsize       = 10,
  angle_col      = 45,
  main           = "Overlap: cell_type vs assigned_celltype"
)


rds_target_file   <- file.path(
  "/Volumes/ProstateCancerEvoMain/dbs/Completed/AllRegions/MMI_WNN",
  paste0(xenium_name, ".raw.annotated.V2.seu.wnn.rds")
)

rds_target_file
#saveRDS(seu, file = rds_target_file)
